# Topic: SQL Duplicate Removal using ROW_NUMBER()

## Definition (30-second explanation)
*   A deduplication technique that uses the `ROW_NUMBER()` window function inside a Common Table Expression (CTE) to assign a unique sequential integer to rows within a specific group.
*   By filtering for `ROW_NUMBER() = 1` in the outer query, you isolate a single representative record per group, effectively removing duplicates.

## Why Interviewers Ask This
*   It tests your understanding of Window Functions and CTEs, which are essential for advanced SQL.
*   It demonstrates your practical data engineering skills, specifically in data cleaning and ETL pipelines.
*   It shows you understand the difference between analyzing data (SELECT) and modifying data permanently (DELETE).

## Core Concepts
*   **CTE (Common Table Expression):** Creates a temporary result set to hold the window function calculations since you cannot filter on window functions directly in a `WHERE` clause.
*   **PARTITION BY:** Defines the grouping logic (e.g., group by `user_id` to find duplicates of the same user).
*   **ORDER BY (within OVER):** Determines the sorting rule within each partition, dictating *which* duplicate gets row number 1 (e.g., `created_at DESC` keeps the newest).

## When to Use
*   Cleaning ETL pipeline bugs, data imports, or double form submissions.
*   When deduplicating entities (like customers) based on natural keys (email, phone).
*   When you need to retain the full row of the most recent (or oldest) event per user.

## Advantages
*   **Flexible Ordering:** You control exactly which record is kept (newest, oldest, highest value).
*   **Preserves Row Integrity:** Unlike `GROUP BY`, you don't have to wrap every other column in an aggregate function.
*   **Safe:** Allows you to preview the deduplicated dataset before running a destructive `DELETE`.

## Limitations
*   More verbose and computationally expensive than a simple `DISTINCT`.
*   Requires a subquery or CTE; cannot be evaluated in a single basic `SELECT ... WHERE` block.

## Common Comparisons
*   **ROW_NUMBER() vs. DISTINCT:** `DISTINCT` only removes exact, 100% identical rows. `ROW_NUMBER()` handles logical duplicates where some columns (like timestamps) might differ.
*   **ROW_NUMBER() vs. RANK():** `ROW_NUMBER()` always gives a unique sequential integer (1, 2, 3). `RANK()` allows ties (1, 1, 3), which fails to isolate a single row if sorting columns have identical values.
*   **ROW_NUMBER() vs. GROUP BY:** `GROUP BY` requires you to aggregate non-grouped columns (`MIN`, `MAX`), making it hard to retrieve the exact original row intact.

## Common Interview Traps
*   **Missing PARTITION BY:** Forgetting this ranks all rows in the table globally, rather than restarting the count for each group.
*   **Destructive Deletes:** Running a `DELETE` statement without first testing the logic with a `SELECT` statement.
*   **Ambiguous Ordering:** Not defining a tie-breaker in the `ORDER BY`, meaning SQL will arbitrarily pick which row gets `rn = 1`.

## Python / SQL Syntax
```sql
-- Step 1: Identify and Rank
WITH ranked_records AS (
    SELECT 
        *, 
        ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY created_at DESC) as rn
    FROM users
)
-- Step 2: Filter out duplicates
SELECT * 
FROM ranked_records 
WHERE rn = 1;

## 45-Second Interview Answer

"To handle duplicate records in SQL, I prefer using a CTE with the ROW_NUMBER() window function. First, I partition the data by the unique identifier, like user_id, and apply an ORDER BY clause—usually on a timestamp—to ensure the most recent record gets row number 1. Then, I query the CTE and filter for where the row number equals 1. I prefer this over SELECT DISTINCT because it allows me to deduplicate based on specific key columns even if other columns, like the update time, vary slightly. It's also much safer because I can preview the exact records before converting it into a DELETE statement."

## Practice Questions:

** Mock Schema and Data:**
```sql
-- Create the table
CREATE TABLE customer_contacts (
    contact_id INT PRIMARY KEY,
    customer_id INT,
    contact_type VARCHAR(50),
    contact_value VARCHAR(100),
    updated_at TIMESTAMP
);

-- Insert sample data (including duplicates)
INSERT INTO customer_contacts (contact_id, customer_id, contact_type, contact_value, updated_at) 
VALUES 
    -- Customer 101: Has duplicate emails, keep contact_id 2
    (1, 101, 'email', 'old.email@test.com', '2026-01-10 08:00:00'),
    (2, 101, 'email', 'new.email@test.com', '2026-08-11 10:00:00'),
    (3, 101, 'phone', '555-0101', '2026-02-15 14:30:00'),
    
    -- Customer 102: Has duplicate phones, keep contact_id 6
    (4, 102, 'email', 'bob@test.com', '2025-11-20 09:15:00'),
    (5, 102, 'phone', '555-0202', '2025-12-01 11:00:00'),
    (6, 102, 'phone', '555-0999', '2026-05-10 16:45:00'),
    
    -- Customer 103: No duplicates
    (7, 103, 'email', 'carol@test.com', '2026-07-01 08:00:00');
```

### Q1: Question: Deduplicate Records with Multiple Keys

**Context:** 
You are working with a CRM database. The `customer_contacts` table contains duplicate entries due to a migration bug. A duplicate is defined as having the same `customer_id` and `contact_type`. 

**Question:** 
Write a SQL query to return a deduplicated view of the table, keeping only the most recently updated record.

**Answer:**
```sql
WITH ranked_records AS (
    SELECT 
        *, -- Always keep the Primary Key!
        ROW_NUMBER() OVER (
            PARTITION BY customer_id, contact_type 
            ORDER BY updated_at DESC
        ) as rnk
    FROM customer_contacts
)
SELECT * 
FROM ranked_records 
WHERE rnk = 1;
```
**Interview Tips:**

**Compound Keys:** Always double-check if a "duplicate" is defined by a single column (like user_id) or a combination of columns (like customer_id + contact_type). Add all defining columns to your PARTITION BY.

**Retain the PK:** Always select the primary key inside your CTE. You will almost always need it if the interviewer asks you to actually execute a DELETE statement.